# Baomai: test the bias warmup rate

**Run in a GPU runtime, from top to bottom.** This notebook prepares a fresh checkout, checks the GPU, runs a short resource trial, trains two models sequentially, and downloads one results ZIP.

Both models start from the original epoch173 checkpoint and use the original reference labels. The only scientific change is `warmup_bias_lr`: **0.1 (control) versus 0.0001 (experiment)**. Both retain 30 epochs, batch 4, image size 1280, FP32, AdamW and seed 42. Keep the same session for both runs.

The original checkpoint remains the working model. The first completed GPU pair is preserved in [PR #16](https://github.com/Siuuuers/dl/pull/16). Use this notebook to reproduce that declared experiment; model selection follows the common evaluation, not training completion or the console's `best.pt` score.

Training source is pinned to [`679b87b38e7b0bca5fbd02fb1519a90fb2379541`](https://github.com/Siuuuers/dl/tree/679b87b38e7b0bca5fbd02fb1519a90fb2379541). See the [full experiment and decision rule](https://github.com/Siuuuers/dl/blob/679b87b38e7b0bca5fbd02fb1519a90fb2379541/docs/gpu-warmup-handoff.md). Cells contain no precomputed outputs and have not been executed on Colab by the author.

## 1. Check the GPU and create a fresh workspace

In Colab, select **Runtime → Change runtime type → GPU**, then run the cells in order. An available CUDA GPU is required. This creates a new dated directory under `/content`; it does not reuse previous model runs. If starting over, rerun this cell to choose a new workspace.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json
import os
import subprocess
import sys
import time

SOURCE_REVISION = "679b87b38e7b0bca5fbd02fb1519a90fb2379541"

def run(command, *, cwd=None, env=None):
    print("+", " ".join(map(str, command)), flush=True)
    return subprocess.run(list(map(str, command)), cwd=cwd, env=env, check=True)

run([sys.executable, "-c", "import torch, torchvision; "
     "assert torch.cuda.is_available(), 'Select a CUDA GPU runtime before continuing.'; "
     "print('PyTorch:', torch.__version__, 'CUDA:', torch.version.cuda); "
     "print('GPU:', torch.cuda.get_device_name(0)); "
     "print('CUDA calculation:', (torch.ones(1, device='cuda:0') * 2).item())"])
run(["nvidia-smi"])
assert sys.version_info >= (3, 11), "Python 3.11 or newer is required."
RUN_BASE = Path("/content") / ("baomai-warmup-" + datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S-%f"))
RUN_BASE.mkdir(parents=True, exist_ok=False)
REPO = RUN_BASE / "dl"
print("Workspace:", RUN_BASE)


## 2. Download the pinned code and original inputs

Only the 157 original training/validation PNGs and epoch173 weights are fetched through Git LFS. Labels and code come from Git. This downloads a fresh input copy directly to the GPU machine; you do not need to upload the old input ZIP.

In [ ]:
lfs = subprocess.run(["git", "lfs", "version"], capture_output=True, text=True)
if lfs.returncode:
    run(["apt-get", "update", "-qq"])
    run(["apt-get", "install", "-y", "-qq", "git-lfs"])
git_env = {**os.environ, "GIT_LFS_SKIP_SMUDGE": "1", "GIT_TERMINAL_PROMPT": "0"}
run(["git", "init", REPO], env=git_env)
run(["git", "remote", "add", "origin", "https://github.com/Siuuuers/dl.git"], cwd=REPO)
run(["git", "fetch", "--depth=1", "origin", SOURCE_REVISION], cwd=REPO, env=git_env)
run(["git", "checkout", "--detach", SOURCE_REVISION], cwd=REPO, env=git_env)
run(["git", "lfs", "install", "--local"], cwd=REPO)
run(["git", "lfs", "pull", "--include=data/images/train/*.png,data/images/val/*.png,weights/cassette_lower_epoch173.pt",
     "--exclude="], cwd=REPO)
actual_revision = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
assert actual_revision == SOURCE_REVISION
assert not subprocess.check_output(["git", "status", "--porcelain"], cwd=REPO, text=True).strip()
print("Verified clean source:", actual_revision)


## 3. Preserve CUDA PyTorch and install the pinned trainer

The installed `torch` and `torchvision` versions are constrained during installation. If this cell reports a dependency conflict, resolve it before training; do not discard the constraints and silently change the GPU runtime. Training uses fresh subprocesses, so it sees the installed packages without importing the trainer into this notebook's kernel.

In [ ]:
from importlib.metadata import version
constraints = RUN_BASE / "torch-constraints.txt"
constraints.write_text(
    ''.join(f'{name}=={version(name)}\n' for name in ('torch', 'torchvision')), encoding='utf-8'
)
run([sys.executable, "-m", "pip", "install", "-c", constraints, "ultralytics==8.4.182", "psutil"])
run([sys.executable, "-c", "import torch, torchvision, ultralytics; "
     "assert ultralytics.__version__ == '8.4.182'; assert torch.cuda.is_available(); "
     "print('Ready:', torch.__version__, torchvision.__version__, ultralytics.__version__, torch.cuda.get_device_name(0))"])
import psutil
HOST_RSS_LIMIT_MIB = min(12000, int(psutil.virtual_memory().available / 2**20 * 0.80))
assert HOST_RSS_LIMIT_MIB >= 6000, "Free more host RAM or select a larger-memory runtime, then rerun this cell."
print("Host-RAM stop budget (not GPU VRAM):", HOST_RSS_LIMIT_MIB, "MiB")
print("Time budgets: 600 seconds for the trial; 21600 seconds per complete run.")


## 4. Define progress reporting and run a short real training trial

The trial uses the lower-bias recipe's full 30-epoch schedule and stops after four batches. It must report actual optimizer updates and a parameter change. It does not produce an eligible model or qualify all later validation/finalization memory costs. Progress appears below; detailed trainer output remains in each run's `train.log`.

In [ ]:
def train_one(folder, recipe, *, trial=False):
    output = RUN_BASE / folder
    command = [sys.executable, "scripts/train_pose_comparison.py",
               "--trial" if trial else "--train", "--arm", "reference", "--recipe", recipe,
               "--device", "0", "--threads", "4", "--trial-batches", "4",
               "--max-seconds", "600" if trial else "21600",
               "--max-rss-mib", str(HOST_RSS_LIMIT_MIB), "--output", str(output)]
    print("Starting:", folder, recipe, flush=True)
    print("Log:", output / "train.log", flush=True)
    process = subprocess.Popen(command, cwd=REPO, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, start_new_session=True)
    descendants = {}

    def remember_descendants():
        try:
            for child in psutil.Process(process.pid).children(recursive=True):
                descendants[child.pid] = child
        except psutil.NoSuchProcess:
            pass

    started = time.monotonic()
    last_notice = -30.0
    try:
        while process.poll() is None:
            remember_descendants()
            elapsed = time.monotonic() - started
            if elapsed - last_notice >= 20:
                progress_file = output / "training-progress.json"
                if progress_file.exists():
                    progress = json.loads(progress_file.read_text())
                    print(f"{folder}: {elapsed:.0f}s; status={progress.get('status')}; "
                          f"completed epochs={len(progress.get('completed_epochs', []))}/30; "
                          f"batches={progress.get('batches_completed', 0)}; "
                          f"updates={progress.get('optimizer_steps', 0)}", flush=True)
                else:
                    print(f"{folder}: {elapsed:.0f}s; verifying/staging inputs", flush=True)
                last_notice = elapsed
            time.sleep(1)
        captured, _ = process.communicate()
        print(captured)
        if process.returncode:
            receipt_file = output / "training-receipt.json"
            if receipt_file.exists():
                failed_receipt = json.loads(receipt_file.read_text())
                for key in ("error", "input_verification_error", "checkpoint_manifest_error"):
                    if failed_receipt.get(key):
                        print(f"{key}: {failed_receipt[key]}", flush=True)
            log = output / "train.log"
            if log.exists():
                print("\n".join(log.read_text(errors="replace").splitlines()[-45:]))
            raise RuntimeError(f"{folder} failed. Preserve its receipt/log and use a new output directory for a retry.")
    except BaseException:
        # Stop this invocation's descendants if the notebook cell is interrupted.
        # Partial runs remain ineligible; no receipt is fabricated here.
        remember_descendants()
        children = list(descendants.values())
        for child in reversed(children):
            try:
                child.terminate()
            except psutil.NoSuchProcess:
                pass
        if process.poll() is None:
            process.terminate()
        _, alive = psutil.wait_procs(children, timeout=5)
        for child in alive:
            try:
                child.kill()
            except psutil.NoSuchProcess:
                pass
        try:
            process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        raise
    receipt = json.loads((output / "training-receipt.json").read_text())
    assert receipt["status"] == ("trial-complete" if trial else "complete")
    assert receipt["optimizer_steps"] > 0 and receipt["optimizer_parameter_change_observed"] is True
    assert receipt["recipe_id"] == recipe
    assert receipt["training_settings"]["epochs"] == 30
    assert (output / "checkpoint-manifest.json").exists() is (not trial)
    return output

TRIAL = train_one("resource-trial", "warmup-bias-lr-1e-4-v1", trial=True)
print("Resource trial passed. This is not a completed model.")


## 5. Train the two reference-label recipes sequentially

Allow this cell to finish both runs in the same session. The original control runs first, then the lower-bias recipe. Each starts freshly from epoch173. Keep this source checkout and environment unchanged. A failed or interrupted run cannot be packaged as a completed experiment.

If the control finished but the warmup run failed, preserve the failure log and diagnose it. To retry only that run in this unchanged session, use a new cell with `WARMUP = train_one("warmup-retry-001", "warmup-bias-lr-1e-4-v1")`, then continue to packaging. If code, environment, or scientific settings changed, start a fresh matched pair instead.

In [ ]:
CONTROL = train_one("control", "paired-finetune-v1")
WARMUP = train_one("warmup", "warmup-bias-lr-1e-4-v1")
for role, folder in (("control", CONTROL), ("warmup", WARMUP)):
    receipt = json.loads((folder / "training-receipt.json").read_text())
    print(role, "epochs:", len(receipt["completed_epochs"]),
          "batches:", receipt["batches_completed"], "optimizer updates:", receipt["optimizer_steps"],
          "GPU:", receipt["runtime"]["device_name"])


## 6. Verify and package the results

The ZIP contains both final-epoch checkpoints and their receipts, settings, input manifests, CSV results and raw logs. The packer rejects different source/runtime identities, changed labels or any extra scientific setting change. Download the ZIP before closing this Colab runtime, then attach it in chat or return it through a new result PR. Original images and intermediate checkpoints do not need to be returned.

In [ ]:
RESULTS_ZIP = RUN_BASE / "baomai-warmup-results.zip"
run([sys.executable, "scripts/package_warmup_results.py", "--control", CONTROL,
     "--warmup", WARMUP, "--output", RESULTS_ZIP], cwd=REPO)
with RESULTS_ZIP.open("rb") as stream:
    digest = hashlib.file_digest(stream, "sha256").hexdigest()
print("Return:", RESULTS_ZIP.name)
print("Bytes:", RESULTS_ZIP.stat().st_size)
print("SHA-256:", digest)


## 7. Download and return the ZIP

Run this download cell again if the browser blocks or interrupts the transfer. It reuses the verified ZIP and does not rerun training or packaging. Send `baomai-warmup-results.zip` back in chat or through a new result PR before closing the runtime.

In [ ]:
assert RESULTS_ZIP.is_file(), "Finish the packaging cell first."
from google.colab import files
files.download(str(RESULTS_ZIP))


### What happens after the ZIP is returned

Evaluate the original epoch173 model, this new control and this warmup model together on the same frozen 20 validation images. At box confidence 0.25, require no increase in missed objects, extra detections, missing visible endpoints, median corner error or p95 corner error, with at least one strict improvement. The warmup model must pass against both M0 and its matched control before the revised recipe is adopted. Full corner-pair and endpoint-availability diagnostics are retained.

Until those results exist, keep the original model. This one-seed development-validation experiment tests the warmup hypothesis; it does not establish independent-test accuracy.